<a href="https://colab.research.google.com/github/stacykeago/Big-Data/blob/main/BDA_Lecture_2_MS_Extension_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Analytics Lecture 2 - MS Extension Lab

**Theme:** Mini clickstream pipeline using Pandas, MapReduce-style chunking, and PySpark.

Run the notebook from top to bottom. Work in pairs/groups and answer the questions at the end.

## Part A - Generate a synthetic clickstream dataset
This simulates customer activity from an online business. It is not real customer data.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200000
channels = ['web', 'mobile_app', 'store', 'social_media', 'email']
categories = ['electronics', 'fashion', 'home', 'books', 'sports']
actions = ['view', 'cart', 'purchase', 'return']

df = pd.DataFrame({
    'event_id': range(1, n + 1),
    'customer_id': np.random.randint(1000, 20000, size=n),
    'channel': np.random.choice(channels, size=n),
    'category': np.random.choice(categories, size=n),
    'action': np.random.choice(actions, size=n, p=[0.55, 0.20, 0.20, 0.05]),
    'device': np.random.choice(['desktop', 'mobile', 'tablet'], size=n),
    'event_time': pd.date_range('2026-01-01', periods=n, freq='min')
})

df['revenue'] = np.where(df['action'] == 'purchase', np.random.gamma(2.0, 45.0, size=n).round(2), 0)
df.to_csv('ms_clickstream_events.csv', index=False)

df.head()

,event_id,customer_id,channel,category,action,device,event_time,revenue
0,1,16795,social_media,electronics,purchase,mobile,2026-01-01 00:00:00,273.27
1,2,1860,social_media,books,view,mobile,2026-01-01 00:01:00,0.00
2,3,6390,store,sports,view,mobile,2026-01-01 00:02:00,0.00
3,4,12964,web,books,view,desktop,2026-01-01 00:03:00,0.00
4,5,12284,social_media,sports,view,mobile,2026-01-01 00:04:00,0.00


# Check for columns and rows

In [2]:
df.shape

(200000, 8)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   event_id     200000 non-null  int64         
 1   customer_id  200000 non-null  int64         
 2   channel      200000 non-null  object        
 3   category     200000 non-null  object        
 4   action       200000 non-null  object        
 5   device       200000 non-null  object        
 6   event_time   200000 non-null  datetime64[ns]
 7   revenue      200000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 12.2+ MB


## Part B - Pandas baseline analysis
Pandas processes this data on one machine. We use it first to understand the logic.

In [4]:
# Total revenue by channel
df.groupby('channel')['revenue'].sum().sort_values(ascending=False)

,revenue
channel,
email,724824.18
mobile_app,724594.58
store,714537.17
web,709006.19
social_media,707735.33


# Highest revenue   : email	724824.18

In [5]:
# Number of purchases by category
df[df['action'] == 'purchase'].groupby('category').size().sort_values(ascending=False)

,0
category,
books,8083
home,8020
fashion,8007
electronics,7875
sports,7823


# Highest number of purchase :  books	8083

In [6]:
# Return rate by category
category_actions = pd.crosstab(df['category'], df['action'])
category_actions['return_rate'] = category_actions['return'] / category_actions.sum(axis=1)
category_actions.sort_values('return_rate', ascending=False)

action,cart,purchase,return,view,return_rate
category,,,,,
electronics,7966,7875,2045,22028,0.051235
sports,8101,7823,2029,22024,0.050754
fashion,7833,8007,2008,22008,0.050381
books,8104,8083,2008,21890,0.050094
home,8091,8020,2002,22055,0.049841


In [7]:
# Revenue by category and channel
pd.pivot_table(df, values='revenue', index='category', columns='channel', aggfunc='sum', fill_value=0)

channel,email,mobile_app,social_media,store,web
category,,,,,
books,145112.87,144830.67,139076.82,149955.58,146636.28
electronics,145733.41,144286.61,138269.68,135932.71,138827.32
fashion,144173.37,142008.80,146520.91,144102.62,146778.50
home,147266.38,152951.48,143588.54,139280.96,142913.99
sports,142538.15,140517.02,140279.38,145265.30,133850.10


## Part C - MapReduce-style chunking simulation
This is not Hadoop, but it demonstrates the same idea: split data, map partial results, reduce/combine final results.

In [8]:
# Split the dataset into 4 chunks, as if 4 workers were processing data
chunks = np.array_split(df, 4)
len(chunks), [chunk.shape[0] for chunk in chunks]

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


(4, [50000, 50000, 50000, 50000])

In [9]:
# Map phase: each chunk calculates partial counts by category
partial_results = []

for i, chunk in enumerate(chunks):
    counts = chunk.groupby('category').size()
    partial_results.append(counts)
    print(f'Chunk {i+1} partial result:')
    print(counts)
    print('-' * 40)

Chunk 1 partial result:
category
books           9965
electronics    10053
fashion         9960
home           10052
sports          9970
dtype: int64
----------------------------------------
Chunk 2 partial result:
category
books          10116
electronics    10002
fashion         9978
home            9886
sports         10018
dtype: int64
----------------------------------------
Chunk 3 partial result:
category
books          10049
electronics     9920
fashion         9883
home           10166
sports          9982
dtype: int64
----------------------------------------
Chunk 4 partial result:
category
books           9955
electronics     9939
fashion        10035
home           10064
sports         10007
dtype: int64
----------------------------------------


In [10]:
# Reduce phase: combine partial results into one final result
final_category_counts = pd.concat(partial_results, axis=1).fillna(0).sum(axis=1).sort_values(ascending=False)
final_category_counts

,0
category,
home,40168
books,40085
sports,39977
electronics,39914
fashion,39856


In [11]:
# Compare with direct Pandas result to verify correctness
direct_category_counts = df.groupby('category').size().sort_values(ascending=False)
comparison = pd.DataFrame({
    'mapreduce_style_count': final_category_counts,
    'direct_pandas_count': direct_category_counts
})
comparison['difference'] = comparison['mapreduce_style_count'] - comparison['direct_pandas_count']
comparison

,mapreduce_style_count,direct_pandas_count,difference
category,,,
home,40168,40168,0
books,40085,40085,0
sports,39977,39977,0
electronics,39914,39914,0
fashion,39856,39856,0


## Part D - PySpark mini-lab
Spark can process similar operations across distributed systems. In Colab we run Spark locally, but the syntax and abstraction are similar to cluster Spark.

In [12]:
!pip install pyspark -q

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum, count, avg, col

spark = SparkSession.builder.appName('BDA_MS_Extension').getOrCreate()
spark_df = spark.read.csv('ms_clickstream_events.csv', header=True, inferSchema=True)
spark_df.show(5)

+--------+-----------+------------+-----------+--------+-------+-------------------+-------+
|event_id|customer_id|     channel|   category|  action| device|         event_time|revenue|
+--------+-----------+------------+-----------+--------+-------+-------------------+-------+
|       1|      16795|social_media|electronics|purchase| mobile|2026-01-01 00:00:00| 273.27|
|       2|       1860|social_media|      books|    view| mobile|2026-01-01 00:01:00|    0.0|
|       3|       6390|       store|     sports|    view| mobile|2026-01-01 00:02:00|    0.0|
|       4|      12964|         web|      books|    view|desktop|2026-01-01 00:03:00|    0.0|
|       5|      12284|social_media|     sports|    view| mobile|2026-01-01 00:04:00|    0.0|
+--------+-----------+------------+-----------+--------+-------+-------------------+-------+
only showing top 5 rows


In [14]:
spark_df.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- category: string (nullable = true)
 |-- action: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- revenue: double (nullable = true)



In [15]:
# Revenue by channel using PySpark
spark_df.groupBy('channel').agg(spark_sum('revenue').alias('total_revenue')).orderBy(col('total_revenue').desc()).show()

+------------+-----------------+
|     channel|    total_revenue|
+------------+-----------------+
|       email|724824.1800000002|
|  mobile_app|724594.5799999993|
|       store|714537.1699999999|
|         web|709006.1900000009|
|social_media|707735.3300000012|
+------------+-----------------+



In [16]:
# Purchases by category using PySpark
spark_df.filter(col('action') == 'purchase').groupBy('category').count().orderBy(col('count').desc()).show()

+-----------+-----+
|   category|count|
+-----------+-----+
|      books| 8083|
|       home| 8020|
|    fashion| 8007|
|electronics| 7875|
|     sports| 7823|
+-----------+-----+



In [17]:
# Average revenue per purchase by channel
spark_df.filter(col('action') == 'purchase').groupBy('channel').agg(avg('revenue').alias('avg_purchase_revenue')).orderBy(col('avg_purchase_revenue').desc()).show()

+------------+--------------------+
|     channel|avg_purchase_revenue|
+------------+--------------------+
|  mobile_app|   90.80132581453624|
|       email|   90.63701137926725|
|social_media|   89.90540269308958|
|         web|   89.73625996709288|
|       store|   88.67425788036732|
+------------+--------------------+



## Part E - Student answer questions
Answer these in your group:

1. How many rows and columns did the dataset contain?
2. Which channel generated the highest revenue? books 8083
3. Which category had the highest number of purchases?
4. What did the MapReduce-style simulation do? Explain map and reduce in your own words.    : 1. we split the data so that we can make processing easier
5. Did the MapReduce-style result match the direct Pandas result? yes they were similar
6. Why might PySpark be preferred over Pandas for very large datasets? pyspark can process the data across distributed systems.
7. If the data arrives continuously from a website/mobile app, which tool would you add and why? pyspark...it carries out batch processing of continous data.
8. Give one business insight from your analysis. Mobile apps generate the most sales, books generate the highest revenue.Marketing measures should be directed towards the other channels to promote more sales on these channels.
